## What is RAG?



![LLM On Unseen data](data/llm_question_ansering.png)








## Full RAG Pipeline

Complete Retrieval-Augmented Generation with LangChain Expresiion Language (LCEL).

In [1]:
# Import necessary libraries for environment variable management and OpenAI API interaction
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv()) # read and set environment variables fromlocal .env file

In [2]:
# Create a document containing information about channel
%mkdir -p data
with open('data/breakdown.txt', 'w') as f:
    f.write("Welcome to BreakDown!\n" +
             "We simplify complex technologies through clear explanations, visual learning, and practical implementations across these areas:\n" +
            "- Latest research in AI, machine learning, and deep learning models\n" +
            "- Research-paper breakdowns and practical coding tutorials\n" +
            "- Applications of AI in autonomous driving, ADAS, and robotics\n" +
            "- Data structures, algorithms, and problem-solving techniques\n" +
            "Subscribe to explore cutting-edge technologies.\n" +
            "Maintained by Mashhood Raza and Aamna Hasan.\n")

### 1. Load the documents and create a vector store for RAG (Retrieval-Augmented Generation) tasks.
#### Load → Split → Embed & Store

#### Step 1 — Load documents
LangChain provides loaders for dozens of file formats. 
They all return a list of Document objects with metadata and page_content.

In [3]:
# Import module for loading text files 
from langchain_community.document_loaders import TextLoader

# Loads the text file and creates a list of documents.
loader= TextLoader("./data/breakdown.txt") #each document has metadata (source, page number, etc.) and page_content (the text).
docs = loader.load() # Load the documents from the text file

print(len(docs)) # Display the number of documents loaded
print(docs[0].metadata) # Display the metadata of the first document
print(docs[0].page_content[:200]) # Display the first 200 characters of the first document

/var/folders/sl/vn5w_6nx0x7bkz_h4h1484600000gn/T/ipykernel_44263/2823286002.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


1
{'source': './data/breakdown.txt'}
Welcome to BreakDown!
We simplify complex technologies through clear explanations, visual learning, and practical implementations across these areas:
- Latest research in AI, machine learning, and dee


#### Step 2 — Chunk the text
LLMs have context limits, and embeddings work best on focused passages — not entire documents. You split documents into smaller chunks before embedding. There are different chunking strategies that can be used.

In [4]:
# Import module for splitting the documents into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Splits the documents into smaller chunks based on the specified chunk size and overlap.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,       # characters per chunk
    chunk_overlap=20,     # overlap to preserve context across chunk boundaries
)
chunks = splitter.split_documents(docs) 

print(len(chunks)) # Display the total number of chunks
print(chunks[0].metadata) # Display the metadata of the first chunk
print(chunks[0].page_content[:50]) # Display the first 50 characters of the first chunk


8
{'source': './data/breakdown.txt'}
Welcome to BreakDown!


#### Step 3 — Embed and store
An embedding model converts each text chunk into a numeric vector. Similar chunks produce similar vectors — this is what enables semantic search.

FAISS (Facebook AI Similarity Search) is an open-source library built by Meta's Fundamental AI Research (FAIR) group. It is specifically designed for efficient similarity search and clustering of dense vectors.

FAISS vs Chroma vs Pinecone: FAISS is great for local development (no server). Chroma is easy to set up with persistence. Pinecone / Weaviate / Qdrant are cloud-hosted for production scale.

In [5]:
# Import modules for creating embeddings and storing them in a vector store
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Instantiate the embeddings model with the specified model name
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Embed all chunks and store in FAISS (in-memory, no server needed)
vector_store = FAISS.from_documents(chunks, embeddings)

# Save to disk so you don't re-embed every time
vector_store.save_local("faiss_index")

### 2. Load and Retrieve the vector store from disk, write prompt, and perform RAG (Retrieval-Augmented Generation) tasks

#### Load & Retrieve → Prompt → RAG Chain

#### Step 1 — Load and Retrieve
A retriever takes a query string and returns the most relevant chunks. The default is similarity search (cosine distance between embeddings).

__MMR — Maximum Marginal Relevance:__ Instead of returning the top-k most similar chunks (which may all say the same thing), MMR balances relevance with diversity. Use search_type="mmr" when your chunks have redundant content.MMR fixes this "echo chamber" effect by picking results iteratively using a two-part scoring formula:
1. Relevance: How closely a candidate item matches the search query.
2. Diversity: How different a candidate item is from the items already chosen

__Multi-query retrieval:__ Generate multiple phrasings of the user's question and retrieve for each. Useful when the user's phrasing doesn't match how the document was written.

```python
from langchain.retrievers import MultiQueryRetriever
retriever = MultiQueryRetriever.from_llm(retriever=base_retriever, llm=llm)
```

In [5]:
# Import modules for loading data from a vector store and use the same embeddings model to ensure compatibility
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Instantiate the embeddings model with the specified model name
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Load the vector store from disk for RAG tasks. 
# The `allow_dangerous_deserialization=True` parameter is used to allow loading of potentially unsafe data, which can be necessary in some cases but should be used with caution.
vector_store = FAISS.load_local("faiss_index", embeddings,
                                allow_dangerous_deserialization=True)

retriever = vector_store.as_retriever(
    search_type="similarity",  # or "mmr"
    search_kwargs={"k": 4},    # return top-4 chunks (Default: 4)
)

docs = retriever.invoke("What is BreakDown?") # Retrieve relevant documents based on the query

#### Step 2 — Prompt for RAG context

Use ChatPromptTemplate to induce the context of the document in your query to LLM.

In [6]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer using the context below. If the answer is not in the context, say "I don't know".
Context: {context}
Question: {question}""")

#### Step 3 — Answer with a RAG chain

Combine the retriever and LLM into an LCEL that fetches context and answers in one call.

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


# Instantiate the ChatOpenAI model with the specified model name and temperature
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Chain without RAG
chain = model | StrOutputParser()
response = chain.invoke("What is BreakDown?")
print(f'Without RAG: {response}')

# Chain with RAG
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt 
    | model 
    | StrOutputParser()
)
rag_response = rag_chain.invoke("What is BreakDown?")
print(f'With RAG: {rag_response}')

Without RAG: "BreakDown" can refer to different concepts depending on the context. Here are a few possible interpretations:

1. **General Definition**: In a general sense, a breakdown refers to a failure or collapse of a system, process, or structure. This could be emotional (e.g., a mental breakdown), mechanical (e.g., a car breakdown), or organizational (e.g., a breakdown in communication).

2. **Psychological Context**: In psychology, a breakdown often refers to a mental or emotional crisis where an individual is unable to cope with stress or overwhelming emotions, leading to a significant decline in mental health.

3. **Business Context**: In business, a breakdown might refer to a failure in operations, such as a breakdown in supply chain logistics or a breakdown in team dynamics.

4. **Data Analysis**: In data analysis, a breakdown can refer to the disaggregation of data into smaller components for detailed analysis, such as breaking down sales figures by region or product type.

